# Advanced Problems with Solutions: `break` and `continue` inside `try...except...finally`

This notebook explores subtle control-flow behavior when `break` and `continue` are used inside loops that also contain `try...except...finally` blocks.

## Learning goals

- Predict execution order with `try`, `except`, `else`, and `finally`
- Understand why `finally` still runs before `break` or `continue` takes effect
- Trace loop behavior precisely
- Recognize best practices and avoid confusing patterns

## Key rule

In Python, the `finally` block runs **no matter how control leaves the `try` statement**: normal completion, exception handling, `return`, `break`, or `continue`.

## Problem 1 - Predict the output of `continue` inside `except`

What is printed by the following code?

In [1]:
for x in [2, 1, 0, -1]:
    print(f"start {x}")
    try:
        value = 10 / x
    except ZeroDivisionError:
        print("except")
        continue
    finally:
        print("finally")
    print("after try")

start 2
finally
after try
start 1
finally
after try
start 0
except
finally
start -1
finally
after try


### Solution 1

The `finally` block runs on every iteration. When `x == 0`, the `except` block runs and then `continue` skips `print("after try")`. But before the loop continues, `finally` still executes.

Expected output:

```text
start 2
finally
after try
start 1
finally
after try
start 0
except
finally
start -1
finally
after try
```

### Why

- For `2`, `1`, and `-1`: division succeeds, `finally` runs, then code after the `try` runs.
- For `0`: exception occurs, `except` runs, `continue` is requested, `finally` runs, then the loop jumps to the next iteration.

## Problem 2 - Predict the output of `break` inside `except`

What is printed by this code?

In [2]:
for x in [3, 2, 1, 0, 5]:
    print(f"item {x}")
    try:
        result = 12 / x
    except ZeroDivisionError:
        print("division error")
        break
    finally:
        print("cleanup")
    print("loop body end")

print("done")

item 3
cleanup
loop body end
item 2
cleanup
loop body end
item 1
cleanup
loop body end
item 0
division error
cleanup
done


### Solution 2

When `x == 0`, the `except` block requests a `break`, but Python still runs `finally` before actually breaking out of the loop.

Expected output:

```text
item 3
cleanup
loop body end
item 2
cleanup
loop body end
item 1
cleanup
loop body end
item 0
division error
cleanup
done
```

### Key point

`break` stops the loop **after** `finally` has executed.

## Problem 3 - Add `else` to the `try` statement

Determine the output and explain when the `else` block runs.

In [3]:
for x in [5, 0, 2]:
    try:
        print("try", x)
        y = 10 / x
    except ZeroDivisionError:
        print("except")
        continue
    else:
        print("else")
    finally:
        print("finally")
    print("after")

try 5
else
finally
after
try 0
except
finally
try 2
else
finally
after


### Solution 3

The `else` block runs only if the `try` block completes without raising an exception.

Expected output:

```text
try 5
else
finally
after
try 0
except
finally
try 2
else
finally
after
```

### Why

- `x = 5`: success -> `else` -> `finally` -> code after try
- `x = 0`: exception -> `except` -> `finally` -> `continue` skips `after`
- `x = 2`: same as first iteration

## Problem 4 - Loop `else` versus `try` `else`

This problem combines a loop `else` with a `try` `else`. Predict the output.

In [4]:
for x in [4, 2, 1]:
    try:
        print(f"trying {x}")
        z = 8 / x
    except ZeroDivisionError:
        print("hit zero")
        break
    else:
        print("safe division")
    finally:
        print("finally block")
else:
    print("loop completed normally")

trying 4
safe division
finally block
trying 2
safe division
finally block
trying 1
safe division
finally block
loop completed normally


### Solution 4

There is no zero in the list, so the loop never breaks. Therefore the loop `else` runs.

Expected output:

```text
trying 4
safe division
finally block
trying 2
safe division
finally block
trying 1
safe division
finally block
loop completed normally
```

### Distinction

- `try ... else`: runs when the `try` block has no exception
- `for/while ... else`: runs when the loop finishes without hitting `break`

## Problem 5 - A `break` in `finally` overrides normal loop flow

Read the code carefully. What happens?

In [5]:
for x in [1, 2, 3]:
    try:
        print(f"try {x}")
    finally:
        print(f"finally {x}")
        break
    print(f"after {x}")

print("finished")

try 1
finally 1
finished


### Solution 5

Expected output:

```text
try 1
finally 1
finished
```

### Why

The `break` inside `finally` terminates the loop immediately after the first iteration's `finally` block completes.

### Best-practice note

Although legal, `break` or `continue` inside `finally` is usually a bad idea because it makes control flow harder to reason about.

## Problem 6 - A `return` meets `finally`

What does this function return?

In [6]:
def compute(items):
    total = 0
    for item in items:
        try:
            if item < 0:
                return total
            total += item
        finally:
            print(f"processed {item}")

    return total

result = compute([3, 4, -2, 10])
print("result =", result)

processed 3
processed 4
processed -2
result = 7


### Solution 6

Expected output:

```text
processed 3
processed 4
processed -2
result = 7
```

### Why

When `item` becomes `-2`, the function prepares to `return total`, which is `7`. Before the function actually returns, the `finally` block runs and prints `processed -2`.

### Key lesson

`finally` also runs before a function returns.

## Problem 7 - Exception in `try`, another exception in `finally`

What exception escapes from this code?

In [7]:
def demo():
    try:
        print("in try")
        1 / 0
    finally:
        print("in finally")
        raise ValueError("from finally")

demo()

in try
in finally


ValueError: from finally

### Solution 7

The code prints:

```text
in try
in finally
```

Then the function raises:

```text
ValueError: from finally
```

### Why

The original `ZeroDivisionError` occurs in the `try`, but the `finally` block raises a new `ValueError`. That new exception replaces the original one as the exception that escapes.

### Best-practice note

Avoid raising new exceptions in `finally` unless you are very deliberate about masking earlier exceptions.

## Problem 8 - Refactor a confusing pattern

The code below works, but it is harder to read than necessary. Rewrite it in a clearer way while preserving behavior.

In [8]:
numbers = [5, 2, 0, 4]
results = []

for n in numbers:
    try:
        results.append(10 / n)
    except ZeroDivisionError:
        continue
    finally:
        print(f"handled {n}")

print(results)

handled 5
handled 2
handled 0
handled 4
[2.0, 5.0, 2.5]


### Solution 8

A clearer refactor keeps the cleanup/logging idea explicit, but avoids putting loop-control logic in the exception block when a normal branch is simpler.

One good version is:

```python
numbers = [5, 2, 0, 4]
results = []

for n in numbers:
    try:
        if n == 0:
            print(f"handled {n}")
            continue
        results.append(10 / n)
    finally:
        if n != 0:
            print(f"handled {n}")

print(results)
```

An even simpler and more Pythonic solution is:

```python
numbers = [5, 2, 0, 4]
results = []

for n in numbers:
    print(f"handled {n}")
    if n == 0:
        continue
    results.append(10 / n)

print(results)
```

### Why this is better

Use exceptions for truly exceptional situations. If zero is an expected case in your logic, a direct conditional is often clearer than catching an exception.

## Problem 9 - Nested loops with `break` inside `except`

Which loop is broken by the `break` statement?

In [9]:
for i in range(2):
    print(f"outer {i} start")
    for j in [2, 1, 0, 3]:
        try:
            print(f"  inner {j}")
            10 / j
        except ZeroDivisionError:
            print("  break inner")
            break
        finally:
            print("  finally inner")
    print(f"outer {i} end")

outer 0 start
  inner 2
  finally inner
  inner 1
  finally inner
  inner 0
  break inner
  finally inner
outer 0 end
outer 1 start
  inner 2
  finally inner
  inner 1
  finally inner
  inner 0
  break inner
  finally inner
outer 1 end


### Solution 9

The `break` only exits the **innermost** loop, which is the inner `for j ...` loop.

Expected output:

```text
outer 0 start
  inner 2
  finally inner
  inner 1
  finally inner
  inner 0
  break inner
  finally inner
outer 0 end
outer 1 start
  inner 2
  finally inner
  inner 1
  finally inner
  inner 0
  break inner
  finally inner
outer 1 end
```

### Key point

Even in nested loops, `finally` runs before the `break` exits the active inner loop.

## Problem 10 - Trace state changes carefully

What is the final value of `log`?

In [10]:
log = []

for x in [1, 0, 2]:
    try:
        log.append(f"try-{x}")
        10 / x
    except ZeroDivisionError:
        log.append(f"except-{x}")
        continue
    else:
        log.append(f"else-{x}")
    finally:
        log.append(f"finally-{x}")
    log.append(f"after-{x}")

log

['try-1',
 'else-1',
 'finally-1',
 'after-1',
 'try-0',
 'except-0',
 'finally-0',
 'try-2',
 'else-2',
 'finally-2',
 'after-2']

### Solution 10

Final value of `log`:

```python
[
    'try-1', 'else-1', 'finally-1', 'after-1',
    'try-0', 'except-0', 'finally-0',
    'try-2', 'else-2', 'finally-2', 'after-2'
]
```

### Why

- `x = 1`: success -> `else` -> `finally` -> `after`
- `x = 0`: exception -> `except` -> `finally` -> `continue` skips `after`
- `x = 2`: success again -> `else` -> `finally` -> `after`

This is a good example of why careful tracing matters in interview-style questions.

## Summary of best practices

1. Use `finally` for cleanup actions that must always happen.
2. Remember that `finally` runs before `break`, `continue`, and `return` take effect.
3. Prefer simple conditionals over exceptions when the case is expected.
4. Avoid putting `break`, `continue`, `return`, or new exceptions inside `finally` unless there is a strong reason.
5. Keep control flow easy to follow; correctness is not enough if the code is hard to maintain.

## Optional self-check challenge

Without running the code, predict the output:

```python
count = 0
while count < 4:
    count += 1
    try:
        print("A", count)
        if count % 2 == 0:
            raise ValueError("even")
    except ValueError:
        print("B", count)
        if count == 2:
            continue
        break
    else:
        print("C", count)
    finally:
        print("D", count)
    print("E", count)
else:
    print("F")
```

Try solving it before revealing the answer below.

### Self-check challenge answer

Expected output:

```text
A 1
C 1
D 1
E 1
A 2
B 2
D 2
A 3
C 3
D 3
E 3
A 4
B 4
D 4
```

Explanation:

- `1`: normal path -> `else`, `finally`, then `E`
- `2`: exception path -> `except`, `continue`, but `finally` still runs
- `3`: normal path again
- `4`: exception path -> `except` chooses `break`, but `finally` still runs before loop exits
- Because the loop ends with `break`, loop `else` does not run.